# Full `training_v1` Baseline + CatBoost Experiment
Run cells top-to-bottom. This notebook is orchestration-only: all dataset validation, training, five protected walk-forward folds, evaluation, reporting, artifact saving, and resume behavior are implemented in the repository's `ml.training` modules. The test split is evaluated only after fitting and is never used for model or threshold selection.

In [ ]:
# Cell 1 — Mount persistent Google Drive storage.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — CONFIGURE THESE VALUES.
REPO_URL = 'https://github.com/YOUR_ORG/YOUR_REPO.git'
REPO_COMMIT = 'PASTE_COMMIT_SHA_CONTAINING_THIS_FRAMEWORK'
REPO_DIR = '/content/qauntify_webV1'
DRIVE_ROOT = '/content/drive/MyDrive/Quantify/training_v1_full_001'
DATASET_ROOT = f'{DRIVE_ROOT}/datasets/training_v1'
EXPERIMENT_ROOT = f'{DRIVE_ROOT}/experiments'
BASELINE_DIR = f'{EXPERIMENT_ROOT}/baseline_v1'
CATBOOST_DIR = f'{EXPERIMENT_ROOT}/catboost_v1'
REPORT_DIR = f'{DRIVE_ROOT}/comparison'

In [ ]:
# Cell 3 — Obtain/update the repository and locate the frozen dataset.
import os, pathlib, subprocess
if not pathlib.Path(REPO_DIR, '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--detach', REPO_COMMIT], check=True)
assert pathlib.Path(DATASET_ROOT, 'training_manifest.json').is_file(), f'Missing frozen dataset: {DATASET_ROOT}'
os.makedirs(EXPERIMENT_ROOT, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

In [ ]:
# Cell 4 — Install the exact shared training dependencies.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-training.txt'], check=True)

In [ ]:
# Cell 5 — Preflight: verifies manifest checksum, feature contract, full main splits, and all five protected folds.
subprocess.run([sys.executable, '-m', 'ml.training.verify',
    '--config', 'ml/configs/catboost_v1.yaml',
    '--dataset-root', DATASET_ROOT], cwd=REPO_DIR, check=True)

In [ ]:
# Cell 6 — Full baselines: 3 main tasks + 3 tasks x 5 walk-forward folds. Safe to rerun.
subprocess.run([sys.executable, '-m', 'ml.training.cli',
    '--config', 'ml/configs/baseline_v1.yaml',
    '--dataset-root', DATASET_ROOT,
    '--experiment-dir', BASELINE_DIR,
    '--resume'], cwd=REPO_DIR, check=True)

In [ ]:
# Cell 7 — Full untuned CatBoost: 3 main tasks + 3 tasks x 5 folds. Safe to rerun after disconnect.
subprocess.run([sys.executable, '-m', 'ml.training.cli',
    '--config', 'ml/configs/catboost_v1.yaml',
    '--dataset-root', DATASET_ROOT,
    '--experiment-dir', CATBOOST_DIR,
    '--resume'], cwd=REPO_DIR, check=True)

In [ ]:
# Cell 8 — Generate the final comparison report from completed artifacts.
subprocess.run([sys.executable, '-m', 'ml.training.compare',
    '--baseline-dir', BASELINE_DIR,
    '--catboost-dir', CATBOOST_DIR,
    '--output-dir', REPORT_DIR], cwd=REPO_DIR, check=True)

In [ ]:
# Cell 9 — Display the concise final report; full JSON remains on Drive.
from pathlib import Path
print(Path(REPORT_DIR, 'comparison_report.md').read_text())

## Resume after a disconnect
1. Reconnect to a Colab runtime. 2. Rerun Cells 1–5 with exactly the same Drive paths. 3. Rerun Cell 6 if its manifest is incomplete. 4. Rerun Cell 7; `--resume` skips completed jobs and CatBoost restores the current job from its Drive snapshot. 5. Run Cells 8–9 only after both experiment manifests report 18 completed jobs. Never add `--smoke` to these full commands.